# 实验5-1：卷积与激活操作的基本实现（学生练习版）

本实验使用 NumPy 手写卷积神经网络中的基础算子，不进行完整网络训练。

实验目标：

1. 理解卷积核滑动和局部乘加过程。
2. 手写零填充（padding）与二维卷积。
3. 观察 padding、stride 和不同卷积核对输出的影响。
4. 手写 ReLU、Sigmoid 和 Tanh 激活函数。
5. 可视化卷积前后的图像与特征图。

> 本实验不调用 PyTorch、TensorFlow 或现成卷积函数。

## 学生任务与完成顺序

本练习只需要补全第 5 节的单通道二维卷积和第 6 节的激活函数。请按照以下顺序完成：

1. **运行第 1～4 节**：理解基础知识，加载 8×8 digits 图像，并确认手写 Padding 能正常运行。
2. **阅读第 5 节实现步骤**：先弄清卷积输入、卷积核、Padding、Stride 和输出尺寸之间的关系。
3. **按 TODO 顺序补全卷积函数**：完成填充、输出尺寸、输出数组、卷积核滑动和局部乘加。
4. **运行第 5.1～5.3 节进行检查**：确认不同 Padding、Stride 和卷积核都能生成正确特征图。
5. **按 TODO 顺序补全三个激活函数**：依次完成 ReLU、Sigmoid 和 Tanh。
6. **运行第 6 节进行检查**：观察三条激活函数曲线，并将激活函数应用到卷积结果。

遇到错误时，应先检查当前 TODO 对应的数组形状和返回值，再继续后面的步骤。

## 1. 卷积基础知识

### 1.1 一个卷积层的计算过程

一个卷积层接收输入图像或上一层输出的特征图，并使用若干个卷积核提取局部特征。其计算过程如下：

1. **填充输入**：根据 Padding 设置，在输入特征图四周补充像素。
2. **选取局部区域**：从输入左上角开始，截取与卷积核大小相同的区域。
3. **局部乘加**：将局部区域与卷积核对应位置的数值相乘，然后将所有乘积相加。
4. **加入偏置**：在乘加结果上加一个偏置，得到输出特征图当前位置的值。
5. **滑动卷积核**：按照 Stride 指定的步长向右移动；到达一行末尾后，再向下移动并继续计算。
6. **生成特征图**：遍历所有位置后，一个卷积核得到一张输出特征图。
7. **使用多个卷积核**：每个卷积核学习不同特征。若卷积层包含多个卷积核，就会输出多张特征图，形成多个输出通道。
8. **执行激活操作**：卷积结果通常还要经过 ReLU 等激活函数，再传递给下一层。

在多通道输入中，一个卷积核会分别处理所有输入通道，再将各通道的计算结果相加。一个输出位置的计算可表示为：

$$
Y_{i,j}=\sum_c\sum_m\sum_n X_{c,i+m,j+n}K_{c,m,n}+b
$$

其中，$c$ 表示输入通道，$K$ 表示卷积核，$b$ 表示偏置。CNN 中通常采用不翻转卷积核的互相关运算，但习惯上仍称为卷积。

### 1.2 Padding 的作用

当卷积核靠近图像边缘时，没有足够的像素参与计算。Padding 在输入四周补充像素，最常见的是补 0。

Padding 主要有以下作用：

- **保留边缘信息**：使卷积核能够更充分地处理图像边缘像素。
- **控制输出尺寸**：适当的 Padding 可以避免特征图在每次卷积后迅速缩小。
- **保持空间尺寸**：使用 3×3 卷积核、Stride 为 1、Padding 为 1 时，输出高宽与输入相同。

对于本实验的 8×8 图像和 3×3 卷积核：

- `padding=0, stride=1` 时，输出为 6×6。
- `padding=1, stride=1` 时，输出仍为 8×8。

### 1.3 Stride 的作用

Stride 表示卷积核每次在水平和垂直方向移动的像素数。

Stride 主要有以下作用：

- **控制采样间隔**：Stride 为 1 时逐像素计算，保留的信息较多；Stride 增大时会跳过部分位置。
- **缩小特征图**：步长越大，输出特征图通常越小。
- **减少计算量**：输出位置减少后，卷积计算次数和后续网络的计算量也随之降低。
- **影响细节保留**：较大的 Stride 能快速降采样，但可能丢失小目标和局部细节。

例如，对 8×8 图像使用 3×3 卷积核，当 `padding=1, stride=2` 时，输出为 4×4。

### 1.4 输出尺寸

卷积输出尺寸由输入尺寸、卷积核大小、Padding 和 Stride 共同决定：

输出尺寸为：

$$
H_{out}=\left\lfloor\frac{H+2P-K_h}{S}\right\rfloor+1,\quad
W_{out}=\left\lfloor\frac{W+2P-K_w}{S}\right\rfloor+1
$$

其中，$H$ 和 $W$ 是输入高宽，$K_h$ 和 $K_w$ 是卷积核高宽，$P$ 是 Padding，$S$ 是 Stride。

## 2. 实验准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

np.random.seed(42)

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("NumPy version:", np.__version__)

## 3. 准备输入图像

本实验直接使用 `scikit-learn` 的 `digits` 数据集。每幅图像大小为 8×8，后续所有 padding、stride、卷积和激活操作都使用同一幅真实手写数字图像。

In [ ]:
digits = load_digits()
digit_image = digits.images[0].astype(np.float64) / 16.0

print("digit label:", digits.target[0])
print("image shape:", digit_image.shape)

plt.figure(figsize=(4, 4))
plt.imshow(digit_image, cmap="gray", interpolation="nearest")
plt.title(f"8x8 digit image, label={digits.target[0]}")
plt.axis("off")

plt.tight_layout()
plt.show()

## 4. 手写零填充

下面不调用 `np.pad`，而是创建一个更大的全零数组，再将原图放到中心位置。

In [ ]:
def zero_padding_manual(image, padding=0):
    """在二维图像四周手写补 0。"""
    image = np.asarray(image, dtype=np.float64)

    if image.ndim != 2:
        raise ValueError("image 必须是二维数组")
    if not isinstance(padding, int) or padding < 0:
        raise ValueError("padding 必须是非负整数")
    if padding == 0:
        return image.copy()

    height, width = image.shape
    padded = np.zeros((height + 2 * padding, width + 2 * padding), dtype=image.dtype)
    padded[padding:padding + height, padding:padding + width] = image
    return padded

In [ ]:
padding_results = [
    zero_padding_manual(digit_image, padding=0),
    zero_padding_manual(digit_image, padding=1),
    zero_padding_manual(digit_image, padding=2),
]

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for padding, (ax, result) in enumerate(zip(axes, padding_results)):
    ax.imshow(result, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"padding={padding}\nshape={result.shape}")
    ax.axis("off")

plt.suptitle("zero padding results")
plt.tight_layout()
plt.show()

## 5. 手写单通道二维卷积

程序先根据 padding 得到扩展图像，再用双重循环控制卷积核的位置。每个输出值都来自局部区域与卷积核的逐元素乘加。

### 卷积函数补全步骤

请严格按照下面的顺序实现：

1. 调用 `zero_padding_manual` 对输入图像进行补零。
2. 从 `kernel.shape` 中取得卷积核的高度和宽度。
3. 检查卷积核是否大于填充后的图像。
4. 根据输入尺寸、卷积核尺寸和 Stride 计算输出高度与宽度。
5. 创建形状为 `(output_h, output_w)` 的全零输出数组。
6. 使用两层循环遍历输出特征图的每一个位置。
7. 根据输出位置和 Stride 计算卷积核在输入图像中的起始坐标。
8. 从填充后的图像中截取与卷积核同样大小的局部区域。
9. 将局部区域与卷积核逐元素相乘、求和并加上偏置，写入输出数组。
10. 所有位置计算完成后返回输出特征图。

完成后可用以下尺寸检查结果：

- `padding=0, stride=1`：输出应为 6×6。
- `padding=1, stride=1`：输出应为 8×8。
- `padding=1, stride=2`：输出应为 4×4。

In [ ]:
def conv2d_single_manual(image, kernel, stride=1, padding=0, bias=0.0):
    """手写单通道二维卷积（互相关形式）。"""
    image = np.asarray(image, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)

    if image.ndim != 2 or kernel.ndim != 2:
        raise ValueError("image 和 kernel 必须都是二维数组")
    if not isinstance(stride, int) or stride <= 0:
        raise ValueError("stride 必须是正整数")

    # TODO 1：调用 zero_padding_manual，得到填充后的图像 padded。
    # TODO 2：读取卷积核的高度 kernel_h 和宽度 kernel_w。
    # TODO 3：检查卷积核是否大于填充后的图像，必要时抛出 ValueError。
    # TODO 4：计算输出特征图的高度 output_h 和宽度 output_w。
    # TODO 5：创建全零输出数组 output。
    # TODO 6：使用 out_y 和 out_x 两层循环遍历所有输出位置。
    # TODO 7：根据 stride 计算当前局部区域左上角的 y 和 x 坐标。
    # TODO 8：从 padded 中截取与卷积核大小相同的 local_region。
    # TODO 9：计算 local_region 与 kernel 的逐元素乘积之和，加 bias 后写入 output。
    # TODO 10：返回 output。

    raise NotImplementedError("请按照 TODO 1～10 补全单通道二维卷积")

### 5.1 查看一个位置的卷积计算

In [ ]:
vertical_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=np.float64)

padded_image = zero_padding_manual(digit_image, padding=1)
region_y, region_x = 4, 4
local_region = padded_image[region_y:region_y + 3, region_x:region_x + 3]
products = local_region * vertical_kernel

print(f"位置 ({region_y}, {region_x}) 的局部区域：\n", local_region)
print("\n卷积核：\n", vertical_kernel)
print("\n逐元素乘积：\n", products)
print("\n该位置的输出：", products.sum())

### 5.2 比较 Padding 和 Stride

In [ ]:
settings = [
    (0, 1),
    (1, 1),
    (1, 2),
]

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, (padding, stride) in zip(axes, settings):
    result = conv2d_single_manual(
        digit_image,
        vertical_kernel,
        padding=padding,
        stride=stride,
    )
    ax.imshow(result, cmap="coolwarm")
    ax.set_title(
        f"padding={padding}, stride={stride}\n"
        f"output shape={result.shape}"
    )
    ax.axis("off")

plt.suptitle("effects of padding and stride")
plt.tight_layout()
plt.show()

### 5.3 使用不同卷积核提取特征

In [ ]:
horizontal_kernel = np.array([
    [-1, -1, -1],
    [ 0,  0,  0],
    [ 1,  1,  1],
], dtype=np.float64)

sharpen_kernel = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0],
], dtype=np.float64)

kernels = [
    ("vertical edge", vertical_kernel),
    ("horizontal edge", horizontal_kernel),
    ("sharpen", sharpen_kernel),
]

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for col, (name, kernel) in enumerate(kernels):
    result = conv2d_single_manual(digit_image, kernel, padding=1)
    axes[0, col].imshow(kernel, cmap="coolwarm")
    axes[0, col].set_title(name + " kernel")
    axes[1, col].imshow(result, cmap="coolwarm")
    axes[1, col].set_title(name + " result")

for ax in axes.ravel():
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. 手写激活函数

卷积是线性运算。激活函数为网络加入非线性，使多层网络能够表示复杂关系。

- ReLU：$f(x)=\max(0,x)$
- Sigmoid：$\sigma(x)=\frac{1}{1+e^{-x}}$
- Tanh：$\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}$

### 激活函数补全步骤

请按照 ReLU、Sigmoid、Tanh 的顺序完成：

1. **ReLU**：先将输入转换为 NumPy 浮点数组；再判断每个元素是否大于 0，正数保持不变，其余位置返回 0。
2. **Sigmoid**：将输入转换为浮点数组；为避免指数溢出，分别处理非负元素和负元素；最后返回范围在 0～1 的结果。
3. **Tanh**：先将输入限制在合理范围内；计算 $e^{2x}$；再根据等价公式 $(e^{2x}-1)/(e^{2x}+1)$ 得到结果。
4. **检查特殊值**：ReLU(-1)=0、ReLU(2)=2、Sigmoid(0)=0.5、Tanh(0)=0。
5. **运行可视化**：确认 ReLU、Sigmoid 和 Tanh 曲线形状符合理论定义。

In [ ]:
def relu_manual(x):
    """手写 ReLU。"""
    # TODO 1：将 x 转换为 NumPy 浮点数组。
    # TODO 2：正数位置返回原值，其他位置返回 0。
    raise NotImplementedError("请补全 ReLU")


def sigmoid_manual(x):
    """手写数值稳定版 Sigmoid。"""
    # TODO 1：将 x 转换为 NumPy 浮点数组，并创建同形状的 result。
    # TODO 2：创建 x >= 0 的布尔掩码 positive。
    # TODO 3：对非负元素计算 1 / (1 + exp(-x))。
    # TODO 4：对负元素先计算 exp(x)，再计算 exp(x) / (1 + exp(x))。
    # TODO 5：返回 result。
    raise NotImplementedError("请补全数值稳定版 Sigmoid")


def tanh_manual(x):
    """使用指数定义手写 Tanh。"""
    # TODO 1：将 x 转换为浮点数组，并使用 np.clip 将其限制在 [-20, 20]。
    # TODO 2：计算 exp_2x = exp(2 * x)。
    # TODO 3：根据 (exp_2x - 1) / (exp_2x + 1) 返回结果。
    raise NotImplementedError("请补全 Tanh")

In [ ]:
x_axis = np.linspace(-6, 6, 400)
functions = [
    (relu_manual, "ReLU"),
    (sigmoid_manual, "Sigmoid"),
    (tanh_manual, "Tanh"),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (function, title) in zip(axes, functions):
    ax.plot(x_axis, function(x_axis), linewidth=2)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axvline(0, color="black", linewidth=0.6)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.grid(alpha=0.25)

plt.suptitle("activation functions")
plt.tight_layout()
plt.show()

### 6.1 将激活函数作用于卷积结果

In [ ]:
conv_result = conv2d_single_manual(digit_image, vertical_kernel, padding=1)
activation_results = [
    ("convolution result", conv_result),
    ("ReLU", relu_manual(conv_result)),
    ("Sigmoid", sigmoid_manual(conv_result)),
    ("Tanh", tanh_manual(conv_result)),
]

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, (title, result) in zip(axes, activation_results):
    image_handle = ax.imshow(result, cmap="coolwarm")
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(image_handle, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 7. 实验练习

1. 修改卷积核中的数值，观察特征图变化。
2. 分别设置 `padding=0、1、2`，验证输出尺寸公式。
3. 分别设置 `stride=1、2、3`，比较输出尺寸与细节。
4. 编写一个均值模糊卷积核，并观察处理结果。
5. 比较卷积结果经过 ReLU、Sigmoid 和 Tanh 后的信息差异。

### 实验总结

本实验手写实现了零填充、二维卷积和三种激活函数，并通过可视化观察了卷积核、padding、stride 和激活操作对输出特征图的影响。